In [1]:
# Schema definition & read csv
from pyspark.sql.types import *
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name
    
# Create the schema for the table
orderSchema = StructType([
    StructField("deck", StringType()),
    StructField("format", StringType()),
    StructField("proxy", StringType()),
    StructField("maindeck", StringType()),
    StructField("count", IntegerType()),
    StructField("card", StringType())
    ])
    
# Import all files from bronze folder of lakehouse
df = spark.read.format("csv").option("header", "true").option("delimiter", ";").schema(orderSchema).load("Files/decks_bronze/*.csv")

    
# Add columns IsFlagged, CreatedTS and ModifiedTS
df = df.withColumn("Created_or_Modified_TS", current_timestamp())

# Display the first 10 rows of the dataframe to preview your data
display(df.head(10))
display(df.count())

StatementMeta(, 149a6343-982a-497b-aa1d-bff7193dc600, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5cba34d7-9e62-4564-a65d-0c2cd5577b3a)

167

In [6]:
# Define the schema for the sales_silver table
    
from pyspark.sql.types import *
from delta.tables import *
    
DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.decks_silver") \
    .addColumn("deck", StringType()) \
    .addColumn("format", StringType()) \
    .addColumn("card", StringType()) \
    .addColumn("proxy", StringType()) \
    .addColumn("maindeck", StringType()) \
    .addColumn("count", IntegerType()) \
    .addColumn("Created_or_Modified_TS", TimestampType()) \
    .execute()

StatementMeta(, edcce313-3994-49ac-a652-c7a362422472, 20, Finished, Available, Finished, False)

In [3]:
# Update existing records and insert new ones based on a condition defined by ..? Check what happens if you use non amd non proxy cards for the same card.
from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/decks_silver')
    
dfUpdates = df
    
deltaTable.alias('silver') \
  .merge(
    dfUpdates.alias('updates'),
    'silver.deck = updates.deck and silver.card = updates.card and silver.maindeck = updates.maindeck and silver.format = updates.format and silver.proxy = updates.proxy'
  ) \
   .whenMatchedUpdate(set =
    {
      "count": "updates.count",
      "Created_or_Modified_TS": "updates.Created_or_Modified_TS" 
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "deck": "updates.deck",
      "format": "updates.format",
      "card": "updates.card",
      "maindeck": "updates.maindeck",
      "proxy": "updates.proxy",
      "count": "updates.count",
      "Created_or_Modified_TS": "updates.Created_or_Modified_TS"
    }
  ) \
  .whenNotMatchedBySourceDelete() \
  .execute()

StatementMeta(, 149a6343-982a-497b-aa1d-bff7193dc600, 6, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.types import *
from delta.tables import *
    
# Create customer_gold dimension delta table
DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.dimcard") \
    .addColumn("CardName", StringType()) \
    .addColumn("CardID", LongType()) \
    .execute()

StatementMeta(, 07b3473c-74d1-42aa-bf3d-29c550665cc3, 5, Finished, Available, Finished)

In [2]:
df = spark.read.table("dbo.decks_silver")


from pyspark.sql.functions import col, split
    
# Create customer_silver dataframe
    
dfdimCard_silver = df.dropDuplicates(["card"]).select(col("Card")) 
    
    
# Display the first 10 rows of the dataframe to preview your data

display(dfdimCard_silver.head(10))
display(dfdimCard_silver.count())

StatementMeta(, 52b8c525-a517-4d08-b759-f7d050ef75a5, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ea87a75-e3c0-4987-9715-c2b498a7f9b4)

81

In [7]:
display(df.head(100))

StatementMeta(, d441ac95-2163-4972-a1d6-bd8206a51793, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d68f119-5e52-47ed-a440-5e936773d0de)

In [4]:
# dfdimCard_temp --> dbo.dimCard
# MAXCardID dort ermitteln
# dann left_anti - join

from pyspark.sql.functions import monotonically_increasing_id, col, when, coalesce, max, lit
    
dfdimCard_temp = spark.read.table("dbo.dimCard")
    
MAXCardID = dfdimCard_temp.select(coalesce(max(col("CardID")),lit(0)).alias("MAXCardID")).first()[0]
    
dfdimCard_gold = dfdimCard_silver.join(dfdimCard_temp,(dfdimCard_silver.Card == dfdimCard_temp.CardName) , "left_anti")
    
dfdimCard_gold = dfdimCard_gold.withColumn("CardID",monotonically_increasing_id() + MAXCardID + 1)

# Display the first 10 rows of the dataframe to preview your data

display(dfdimCard_gold.head(10))
display(dfdimCard_gold.count())

StatementMeta(, 52b8c525-a517-4d08-b759-f7d050ef75a5, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 75c7ad2a-637b-4ec5-b90f-7d5cacbb0d71)

0

In [3]:
# insert new card values into dim_card.

from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/dimcard')
    
dfUpdates = dfdimCard_gold
    
deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.CardName = updates.Card'
  ) \
   .whenMatchedUpdate(set =
    {
          
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "CardName": "updates.Card",
      "CardID": "updates.CardID"
    }
  ) \
  .execute()

StatementMeta(, 9847e938-0542-43ad-bf43-47893957ae01, 88, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.types import *
from delta.tables import *

## fact_decks create table

DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.factdecks") \
    .addColumn("CardID", LongType()) \
    .addColumn("deck", StringType()) \
    .addColumn("card", StringType()) \
    .addColumn("maindeck", StringType()) \
    .addColumn("count", IntegerType()) \
    .execute()

StatementMeta(, 44e2b0d1-5d22-41f6-8ad7-348b73966e98, 4, Finished, Available, Finished, False)

In [5]:
# Load data to the dataframe as a starting point to create the gold layer
df = spark.read.table("dbo.decks_silver")

dfdimcard_temp = spark.read.table("dbo.dimCard")

dffact_decks = df.alias("df1").join(dfdimcard_temp.alias("df2"),(df.card == dfdimcard_temp.CardName))
#display(dfdimcard_temp.head(10))
#display(df.head(10))
display(dffact_decks.head(100))

StatementMeta(, 004cf587-3353-4f5e-83d6-42379cc5ad5e, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 490128ca-cc87-4159-975a-b039b53c9e2b)

In [6]:
from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/factdecks')
            
dfUpdates = dffact_decks 
            
deltaTable.alias('gold') \
  .merge(
        dfUpdates.alias('updates'),
        'gold.deck = updates.deck AND gold.card = updates.card'
        ) \
.whenMatchedUpdate(set =
    {
          
    }
  ) \
.whenNotMatchedInsert(values =
         { 
           "CardID": "updates.CardID",
           "deck": "updates.deck",
           "card": "updates.card",
           "maindeck": "updates.maindeck",
           "count": "updates.count"
          }
          ) \
.whenNotMatchedBySourceDelete() \
.execute()

StatementMeta(, 004cf587-3353-4f5e-83d6-42379cc5ad5e, 10, Finished, Available, Finished, False)